# Structured Outputs with Pydantic

Every query so far has returned a free-text string — great to read, unusable to plug directly into a database row, an API call, or a downstream function that expects specific fields. LlamaIndex can force an LLM's output into a typed **Pydantic model** instead, so `response.response` is an actual Python object with real fields, not a string you'd have to parse yourself.


**Step 1 — Set up.** Configures the usual LLM and embedding model before we build anything.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from the HTTP client and LlamaIndex itself.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads .env into os.environ so OPENAI_API_KEY is available to the clients below.
load_dotenv()

Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Define a schema and force the LLM to fill it in.** `AnimeProfile` is a normal Pydantic model; wrapping an LLM with `.as_structured_llm(AnimeProfile)` makes every completion from that LLM come back as an actual `AnimeProfile` instance instead of free text.


In [3]:
from pydantic import BaseModel, Field
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.openai import OpenAI


# Each Field's description is sent to the LLM as part of the extraction prompt —
# it's not just documentation, it directly guides what the model fills in.
class AnimeProfile(BaseModel):
    title: str = Field(description="The anime's title")
    studio: str = Field(description="The animation studio")
    protagonist: str = Field(description="The main character's name")
    signature_technique: str = Field(description="The protagonist's signature technique or ability")
    first_aired_year: int = Field(description="The year the anime first aired")


documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)

# Wrapping an LLM with a Pydantic class turns every completion into that type.
# gpt-4o-mini is used instead of the project-wide gpt-4.1-nano default because
# reliable schema adherence benefits from a more capable model.
structured_llm = OpenAI(model="gpt-4o-mini").as_structured_llm(AnimeProfile)
structured_query_engine = index.as_query_engine(llm=structured_llm)

for title in ("Solo Leveling", "Death Note", "Naruto"):
    response = structured_query_engine.query(f"Extract a profile for the {title} anime.")
    profile: AnimeProfile = response.response  # a real AnimeProfile instance, not a string
    print(f"type: {type(profile).__name__}")
    print(profile.model_dump())  # dumps the Pydantic model to a plain dict
    print(profile.title)
    print()

type: AnimeProfile
{'title': 'Solo Leveling', 'studio': 'A-1 Pictures', 'protagonist': 'Sung Jin-Woo', 'signature_technique': 'Shadow Monarch', 'first_aired_year': 2024}
Solo Leveling

type: AnimeProfile
{'title': 'Death Note', 'studio': 'Studio Madhouse', 'protagonist': 'Light Yagami', 'signature_technique': 'Death Note', 'first_aired_year': 2006}
Death Note

type: AnimeProfile
{'title': 'Naruto', 'studio': 'Studio Pierrot', 'protagonist': 'Naruto Uzumaki', 'signature_technique': 'Rasengan', 'first_aired_year': 2002}
Naruto



### Summary

- `llm.as_structured_llm(SomeModel)` forces every completion from that LLM into an instance of `SomeModel` — `response.response` is a real Pydantic object, not text you'd need to regex or JSON-parse yourself.
- Field `description`s aren't just documentation — they're part of what gets sent to the LLM to guide extraction, so writing them clearly directly improves extraction accuracy.
- `gpt-4o-mini` is used here rather than the project-wide `gpt-4.1-nano` default because reliable schema adherence benefits from a more capable model, the same reasoning used for the router and agent episodes.
